# Relational databases and first queries

This notebook covers the first block of the session. Suppose a housing analyst of the city asks three questions: how many Airbnb listings are there in each district, what does a night cost there, and which listings are actually booked? The answers are spread over several tables: one row per listing, one row per listing and night of the coming year, one row per listing and month with the number of reviews. A relational database keeps such tables linked by keys, and SQL is the language analysts use to get answers out of them.

No prior knowledge of databases is assumed. The course uses **SQLite**: the whole database is one file, and Python includes it as the module `sqlite3`, so nothing has to be installed. The SQL in this notebook is standard SQL; where SQLite differs from server databases such as PostgreSQL, the text says so.

All examples use the course case study, the Berlin snapshot of Inside Airbnb (26 June 2026), in the file `case-study/data/airbnb.sqlite`:

- `listings`: 12,776 listings with host, district, room type, guests, price, minimum stay, reviews and registration status;
- `calendar`: availability of each listing for each of the next 365 nights (4.7 million rows);
- `reviews`: number of reviews per listing and month since 2009.

```mermaid
flowchart LR
    Q["Business question"] --> S["SQL query"]
    S --> DB[("SQLite file<br/>listings, calendar,<br/>reviews")]
    DB --> R["Small result table"]
    R --> P["pandas<br/>in Python"]
    P --> A["Chart, model, report"]
```

> **Note.** The database file is built from the case-study Parquet files by [`make_airbnb_sqlite.py`](make_airbnb_sqlite.py). The code cell in the section on SQLite runs it automatically if the file is missing; you need `case-study/prepare_airbnb.py` to have run once.

In [1]:
# Run the code from the repository root, where the case-study paths start.
import os
from pathlib import Path

os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir()))

## What a relational database is: tables, rows, columns and data types

### Concept

A **relational database** stores data in **tables** (also called relations). Each **row** is one record, for example one listing; each **column** has a name and a **data type** (whole number, decimal number, text). The set of table definitions is the **schema**. The idea goes back to Edgar F. Codd (1970), who proposed that data should be described by tables and queried by their content, not by the way they are stored on disk. A program that stores such tables and answers queries on them is a **database management system** (DBMS); SQLite, PostgreSQL, MySQL and Microsoft SQL Server are examples.

A small example that you can follow by hand (the ids are invented; the real ones have up to 19 digits):

| listings |  |  |  |
|---|---|---|---|
| **id** | district | room_type | price |
| 1 | Mitte | Entire home/apt | 120 |
| 2 | Neukölln | Private room | 45 |
| 3 | Mitte | Entire home/apt | NULL |

| reviews |  |  |
|---|---|---|
| **listing_id** | **month** | n_reviews |
| 1 | 2026-04-01 | 3 |
| 1 | 2026-05-01 | 2 |
| 2 | 2026-05-01 | 1 |

`NULL` marks a value that is missing or unknown: listing 3 shows no price. The last section of this notebook is about NULL.

**Data types in SQLite.** SQLite stores every value in one of five **storage classes**: `NULL`, `INTEGER`, `REAL` (a decimal number), `TEXT` and `BLOB` (raw bytes). It has no separate type for dates or for true/false:

- dates are stored as text in the form `'YYYY-MM-DD'`, which sorts in date order, and SQLite's date functions (`date()`, `strftime()`, `julianday()`) read this form;
- true/false is stored as the integers 1 and 0. A comparison such as `price > 100` also returns 1 or 0.

A column declared `INTEGER` in SQLite prefers integers but, unlike most databases, also accepts other values (*flexible typing*); a table declared `STRICT` rejects them. PostgreSQL has separate `DATE`, `BOOLEAN` and many more types, and always checks them.

### Why it matters

A table with named, typed columns is a contract: every row has the same fields, and every program that reads the table can rely on that. Spreadsheets and CSV files have no such contract; one cell of text in a column of prices is enough to break a calculation.

### In practice

- **Stack Exchange Data Explorer** lets anyone query the public Stack Overflow database with SQL; questions, answers, users and votes are separate tables.
- **Wikimedia** offers public read-only copies of the Wikipedia databases through the Quarry service; pages, revisions and users are related tables.
- **Inside Airbnb** publishes each city as separate files (listings, calendar, reviews); the course database turns them into three tables.

## Primary keys, foreign keys and relationships

### Concept

A **primary key** is a column, or a combination of columns, whose value identifies each row uniquely and is never empty. In the case study, `id` identifies a listing. The review table has a **composite key**: the pair (`listing_id`, `month`) identifies a row, because each listing has at most one row per month. The calendar has the key (`listing_id`, `date`).

A **foreign key** is a column that refers to the primary key of another table. `calendar.listing_id` and `reviews.listing_id` point to `listings.id`. This expresses a **relationship**: one listing has many calendar nights and many review months, each of which belongs to exactly one listing (a *one-to-many* relationship). The database can enforce it: a review row for a listing that does not exist is rejected.

Storing each fact once, in one table, is called **normalisation**. The district of a listing is stored once in `listings`, not repeated in each of its 365 calendar rows. If a listing moves to another district, one row changes, and no copy can contradict another.

```mermaid
classDiagram
    direction LR
    class listings {
        id : INTEGER  PK
        host_id : INTEGER
        district : TEXT
        room_type : TEXT
        price : REAL
        minimum_nights : INTEGER
        license_status : TEXT
    }
    class calendar {
        listing_id : INTEGER  FK
        date : TEXT
        available : INTEGER 0/1
    }
    class reviews {
        listing_id : INTEGER  FK
        month : TEXT
        n_reviews : INTEGER
    }
    listings "1" --> "0..*" calendar : has nights
    listings "1" --> "0..*" reviews : has review months
```

**Following a relationship.** To answer "how many reviews did the listings in Mitte receive?", each review row needs the district of its listing: the database pairs the rows of two tables whose keys match. This operation is a **join**; block 2 teaches its syntax. Three kinds matter, shown on the small tables above:

- an **inner join** returns one row for each pair of matching rows: listing 1 with its two review months and listing 2 with its one, 3 rows; listing 3 has no partner and disappears;
- a **left join** returns the same pairs and, in addition, every row of the left table without a partner, with NULL in the columns of the right table: 4 rows;
- a **cross join** pairs every row of one table with every row of the other, whether the keys match or not: 3 × 3 = 9 rows.

### Why it matters

Keys make the relationship between tables explicit, and the database uses them to guard data quality at the moment data are written: a review row for an unknown listing, or the same night of a listing twice, is rejected immediately instead of being discovered months later in an analysis. The database also builds an **index** (a sorted lookup structure) on every primary key, which makes lookups and joins by key fast. Without normalisation, the same fact is stored many times, and copies drift apart.

### How it works in Python

The case-study tables come from Parquet files, which have no keys. The following check tests the key properties with pandas (Session 1): the primary keys are unique, and every foreign-key value has a partner.

In [2]:
import pandas as pd

listings = pd.read_parquet("case-study/data/airbnb/listings.parquet", columns=["id", "district"])
calendar = pd.read_parquet("case-study/data/airbnb/calendar.parquet", columns=["listing_id", "date"])
reviews = pd.read_parquet("case-study/data/airbnb/reviews_monthly.parquet")

print("listings.id unique:", listings["id"].is_unique)
print("(listing_id, date) duplicated in calendar:", calendar.duplicated(["listing_id", "date"]).any())
# foreign keys: does every calendar and review row refer to an existing listing?
for name, table in [("calendar", calendar), ("reviews", reviews)]:
    unknown = ~table["listing_id"].isin(listings["id"])
    print(f"{name}: {unknown.sum()} rows of {table.loc[unknown, 'listing_id'].nunique()} unknown listings")

listings.id unique: True
(listing_id, date) duplicated in calendar: False


calendar: 28835 rows of 79 unknown listings
reviews: 499 rows of 57 unknown listings


The foreign key fails for 79 listings in the calendar (28,835 rows, exactly 365 per listing) and 57 listings in the review table. Inside Airbnb scrapes listings, calendars and reviews separately, and the listings were scraped between 26 June and 3 July 2026: a listing that went offline between two scrapes appears in one file but not in the other. A real foreign key would have rejected these rows, which is exactly the point: the database makes you decide what to do with them (keep them in a separate table, or drop them and document it) instead of letting a join lose them silently. The course database keeps all rows, so that you can find them with SQL in block 2.

### In practice

- **Online shops** keep customers, orders and products in separate tables; an order row refers to a customer id and a product id, so that a change of address is made once.
- **Data quality checks** in finance and retail look for records that refer to a deleted customer or a discontinued product, exactly like the calendar rows of vanished listings here.

> **Warning.** A file is not a database. A Parquet or CSV file has no keys and no constraints: nothing stops a duplicated `id` or a calendar row for a listing that is not in the listings file. When you work with files, check the key properties yourself (as above) and repeat the check after each update.

## SQLite: a database in one file

### Concept

Most database systems run as a **server**: a separate program that listens on a network port, and clients connect to it with a host name, user and password. **SQLite** is different: it is a library that runs inside your program, and the whole database (all tables, keys and indexes) is one ordinary file. There is nothing to install or start; Python includes it as the module `sqlite3`.

| | SQLite | Server databases (PostgreSQL, MySQL, ...) |
|---|---|---|
| Where the data live | one file on your disk | managed by the server |
| Setup | none | install and run a server, create users |
| Many users writing at once | one writer at a time | yes |
| Typical use | apps, phones, browsers, analysis on one machine, small teams | shared operational data of an organisation |

You can open the file in three ways:

1. **From Python** with `sqlite3.connect("file.sqlite")`, which returns a **connection**; this course mostly uses this way;
2. in a graphical **database browser**, such as [DB Browser for SQLite](https://sqlitebrowser.org/) (free; open the file, then type SQL in the tab "Execute SQL");
3. with the command-line program `sqlite3`, which is preinstalled on macOS and can be downloaded from [sqlite.org](https://www.sqlite.org/download.html) for other systems.

The SQL you learn here is standard SQL and runs, with few changes, on PostgreSQL and other server databases.

### Why it matters

A database you can copy like any file is ideal for learning and for a team project: everyone gets the same data, the queries are the same as on a server, and nothing breaks because a server is not running. The limits matter only when many people write at the same time; then an organisation uses a server database, and the SQL stays the same.

### How it works in Python

The next cell builds the course database if it does not exist yet (a few seconds), opens a connection and asks SQLite which tables the file contains. Every SQLite file has a built-in table `sqlite_master` that lists its tables and indexes; `PRAGMA table_info(listings)` lists the columns of one table.

In [3]:
import sqlite3
import subprocess
import sys

import pandas as pd

DB = Path("case-study/data/airbnb.sqlite")
if not DB.exists():                     # build the course database once from the Parquet files
    subprocess.run([sys.executable, "sessions/03-sql/workbooks/make_airbnb_sqlite.py"], check=True)

con = sqlite3.connect(DB)               # a connection to the database file
print("SQLite version", sqlite3.sqlite_version)
print(con.execute("SELECT name, type FROM sqlite_master").fetchall())

SQLite version 3.53.4
[('listings', 'table'), ('calendar', 'table'), ('reviews', 'table')]


In [4]:
def q(sql, params=()):
    """Run an SQL query on the course database and return the result as a pandas DataFrame."""
    return pd.read_sql_query(sql, con, params=params)


q("PRAGMA table_info(listings)")[["name", "type", "notnull", "pk"]]

,name,type,notnull,pk
0,id,INTEGER,0,1
1,name,TEXT,0,0
2,host_id,INTEGER,1,0
3,district,TEXT,1,0
4,neighbourhood,TEXT,0,0
5,latitude,REAL,0,0
6,longitude,REAL,0,0
7,room_type,TEXT,1,0
8,accommodates,INTEGER,0,0
9,bedrooms,INTEGER,0,0


`con.execute(...)` runs one statement and `.fetchall()` returns the result rows as a list of tuples. The small helper `q` returns the result as a pandas DataFrame instead, which is easier to read; block 2 explains both in detail. The column `pk` marks `id` as the primary key of `listings`.

### In practice

- According to its developers, SQLite is probably the most widely deployed database engine: it is built into every Android and iOS phone, into Firefox, Chrome and Safari, and into many desktop programs, which keep their settings and history in SQLite files.
- The **US Library of Congress** lists SQLite files among its recommended formats for preserving datasets.
- **Data teams** ship small reference datasets as one SQLite file, so that colleagues can query them without access to a server.

> **Tip.** A database browser is useful for a first look at the tables and for trying a query. For work that must be repeated, write the query into a notebook or a script, as in this course.

## Creating tables and loading data: CREATE TABLE, INSERT, COPY

### Concept

`CREATE TABLE` defines a table: its columns, their types and its **constraints**, the rules every row must satisfy.

- `PRIMARY KEY` declares the key; `NOT NULL` forbids missing values;
- `CHECK (condition)` adds a rule for the values, for example `CHECK (price > 0)`;
- `REFERENCES other_table (column)` declares a foreign key.

`INSERT INTO table VALUES (...)` adds rows. A statement that violates a constraint fails with an error message.

Loading many rows one `INSERT` at a time is slow. Every database has a faster way: PostgreSQL has the command `COPY table FROM 'file.csv'`; the SQLite command-line program has `.import file.csv table`; from Python, `executemany` sends many rows with one statement, and pandas writes a whole DataFrame with `DataFrame.to_sql(name, con)`. The course database was built this way by [`make_airbnb_sqlite.py`](make_airbnb_sqlite.py).

> **Note.** For historical reasons SQLite checks foreign keys only when the connection has switched them on with `PRAGMA foreign_keys = ON`. Do this right after connecting whenever you rely on foreign keys.

### Why it matters

Constraints turn silent data problems into loud errors at the moment data are written. A rule written once in the schema protects every program that writes to the table, including the scripts of your teammates.

### How it works in Python

The example creates the two small tables from above in a database that lives only in memory (`":memory:"`), then tries three invalid rows. `executescript` runs several statements at once.

In [5]:
demo = sqlite3.connect(":memory:")       # a temporary database that disappears when the connection closes
demo.execute("PRAGMA foreign_keys = ON")
demo.executescript("""
CREATE TABLE listings (
    id        INTEGER PRIMARY KEY,                 -- one row per listing
    district  TEXT NOT NULL,
    room_type TEXT NOT NULL CHECK (room_type IN ('Entire home/apt', 'Private room', 'Hotel room', 'Shared room')),
    price     REAL CHECK (price > 0)               -- NULL allowed: no price shown
);
CREATE TABLE reviews (
    listing_id INTEGER NOT NULL REFERENCES listings (id),   -- foreign key
    month      TEXT NOT NULL,
    n_reviews  INTEGER NOT NULL CHECK (n_reviews > 0),
    PRIMARY KEY (listing_id, month)                          -- composite key
);
INSERT INTO listings VALUES (1, 'Mitte', 'Entire home/apt', 120), (2, 'Neukölln', 'Private room', 45),
                            (3, 'Mitte', 'Entire home/apt', NULL);
INSERT INTO reviews VALUES (1, '2026-04-01', 3), (1, '2026-05-01', 2), (2, '2026-05-01', 1);
""")

invalid = {
    "unknown listing": "INSERT INTO reviews VALUES (9, '2026-05-01', 1)",
    "same month twice": "INSERT INTO reviews VALUES (1, '2026-05-01', 4)",
    "unknown room type": "INSERT INTO listings VALUES (4, 'Mitte', 'Castle', 300)",
}
for label, statement in invalid.items():
    try:
        demo.execute(statement)
    except sqlite3.IntegrityError as error:
        print(f"{label:18} rejected: {error}")

unknown listing    rejected: FOREIGN KEY constraint failed
same month twice   rejected: UNIQUE constraint failed: reviews.listing_id, reviews.month
unknown room type  rejected: CHECK constraint failed: room_type IN ('Entire home/apt', 'Private room', 'Hotel room', 'Shared room')


The database rejects each invalid row, and the message says which kind of rule failed. Many rows at once, as a loading script would send them:

In [6]:
new_rows = [(4, "Pankow", "Private room", 60.0), (5, "Mitte", "Shared room", None)]
demo.executemany("INSERT INTO listings VALUES (?, ?, ?, ?)", new_rows)   # the ? are filled from each tuple
demo.commit()                                                            # make the changes permanent
print(demo.execute("SELECT COUNT(*) FROM listings").fetchone())

(5,)


The question marks are **placeholders**: SQLite fills them with the values of each tuple. Block 2 explains why values should always be passed this way. `commit()` ends the **transaction**: until then, the new rows are visible only to this connection and would be lost if the program stopped.

### In practice

- **Data engineering teams** keep the `CREATE TABLE` statements of a project in version-controlled `.sql` files and review changes in pull requests, like any other code.
- **Public Health England** lost about 16,000 positive COVID-19 test results in October 2020 because a manual step used an old Excel file format whose row limit was exceeded; a loading script with row-count checks would have caught it.

> **Caution.** `DataFrame.to_sql(..., if_exists="replace")` drops the table and recreates it **without** primary keys or constraints. To keep them, create the table with `CREATE TABLE` first and load with `if_exists="append"`, as `make_airbnb_sqlite.py` does for `listings`.

## Queries: SELECT, WHERE, ORDER BY, LIMIT

### Concept

A query describes the result you want, not the steps to compute it; the database chooses how. The basic clauses are:

- `SELECT` names the columns (or expressions) of the result; `*` means all columns; `AS` gives a column a new name;
- `FROM` names the table;
- `WHERE` gives a condition that each row must satisfy;
- `ORDER BY` sorts the result (`DESC` for descending);
- `LIMIT` keeps only the first *n* rows.

Conditions combine with `AND`, `OR` and `NOT`. `IN (...)` tests membership in a list, `BETWEEN a AND b` a range (both ends included), and `LIKE` a text pattern, where `%` stands for any sequence of characters. `SELECT DISTINCT` removes duplicate rows. Text values are written in single quotes: `'Mitte'`.

Although we write `SELECT` first, the database evaluates the clauses in a different **logical order**:

```mermaid
flowchart LR
    F["1 FROM"] --> W["2 WHERE"] --> G["3 GROUP BY"] --> H["4 HAVING"] --> S["5 SELECT"] --> O["6 ORDER BY"] --> L["7 LIMIT"]
```

`GROUP BY` and `HAVING` follow in block 2. The order explains, for example, why `ORDER BY` can use a name defined with `AS` in `SELECT`: it runs afterwards.

By hand: in the small tables above, `SELECT id FROM listings WHERE district = 'Mitte' ORDER BY id` keeps listings 1 and 3 and returns them in this order.

### Why it matters

Filtering in the database moves only the rows you need into Python. With 4.7 million calendar rows, `SELECT *` followed by filtering in pandas wastes memory and time. Writing the condition out also makes the question precise: "affordable short stays for a family" becomes `room_type = 'Entire home/apt' AND minimum_nights < 28 AND accommodates >= 4 AND price <= 150`.

### How it works in Python

The best-rated entire flats in Neukölln under €100 a night that can be booked for less than 28 nights and have at least 50 reviews. Listing titles are not shown on purpose: hosts sometimes write their own name into the title, and an analysis does not need it.

In [7]:
q("""
SELECT id, neighbourhood, price, number_of_reviews, review_scores_rating
FROM listings
WHERE district = 'Neukölln' AND room_type = 'Entire home/apt'
  AND minimum_nights < 28 AND price < 100 AND number_of_reviews >= 50    -- all conditions must hold
ORDER BY review_scores_rating DESC, number_of_reviews DESC             -- best rated first; reviews break ties
LIMIT 3
""")

,id,neighbourhood,price,number_of_reviews,review_scores_rating
0,31382644,Schillerpromenade,92.5,374,4.91
1,237670,Buckow Nord,68.5,91,4.84
2,190448,Buckow Nord,85.0,87,4.77


In [8]:
print(q("""
SELECT COUNT(*) AS n FROM listings
WHERE room_type = 'Entire home/apt' AND minimum_nights < 28 AND price BETWEEN 50 AND 100
"""))
# LIKE ignores upper and lower case for the letters A-Z in SQLite (PostgreSQL: ILIKE)
print(q("SELECT COUNT(*) AS n FROM listings WHERE name LIKE '%balcony%' OR name LIKE '%balkon%'"))
print(q("SELECT DISTINCT room_type FROM listings ORDER BY room_type"))
q("SELECT id, district, room_type, minimum_nights FROM listings ORDER BY minimum_nights DESC LIMIT 3")

     n
0  336
     n
0  711
         room_type
0  Entire home/apt
1       Hotel room
2     Private room
3      Shared room


,id,district,room_type,minimum_nights
0,6670861,Neukölln,Entire home/apt,1125
1,584757,Pankow,Private room,1000
2,6704144,Pankow,Private room,1000


`COUNT(*)` counts the rows that pass the `WHERE` condition; block 2 introduces it with the other aggregate functions. A minimum stay of 1,125 nights, three years, is not a holiday rental; Session 4 treats such values.

### In practice

- **Property portals** turn the search form (district, number of rooms, maximum rent) into exactly such filter conditions before they sort the result by date or price.
- **Data engineers** check every nightly load with small queries such as "how many orders arrived yesterday?".

> **Caution.** Without `ORDER BY`, the order of rows is not defined. `LIMIT 10` without `ORDER BY` returns *some* ten rows, which may change between runs or database versions.

## NULL values

### Concept

**NULL** means "missing" or "unknown". It is not zero and not the empty text `''`. SQL treats it with **three-valued logic**: a condition is true, false or unknown.

- Any comparison with NULL is unknown: `price > 100` is unknown for a listing without a price, and so is `price = NULL`. `WHERE` keeps only rows whose condition is *true*, so rows with unknown conditions disappear from both `WHERE price > 100` and `WHERE NOT (price > 100)`.
- To test for NULL, write `price IS NULL` or `price IS NOT NULL`.
- Arithmetic and text operations with NULL give NULL: `price * 2` is NULL when the price is.
- `COALESCE(price, 0)` returns the first value that is not NULL, here 0 for missing prices. Use it only when "missing" really means zero.
- Counting: `COUNT(*)` counts rows; `COUNT(price)` counts only the rows where `price` is not NULL. Functions such as `AVG(price)` skip NULLs.
- Sorting: SQLite sorts NULL before every other value (first in ascending order, last in descending order). PostgreSQL does the opposite. `ORDER BY price DESC NULLS LAST` makes the order explicit in both.

By hand, with the three small listings: `WHERE price < 100` keeps only listing 2; `WHERE price >= 100` keeps only listing 1; listing 3 is in neither result. `COUNT(*)` is 3, `COUNT(price)` is 2, and the average price is (120 + 45) / 2 = 82.5, not (120 + 45 + 0) / 3.

### Why it matters

A third of the listings in the case study show no price, so NULL changes almost every answer about prices. A filter that forgets NULL silently changes the population a number describes, and a count over a column instead of over rows silently answers a different question.

### How it works in Python

In [9]:
print(q("""
SELECT (SELECT COUNT(*) FROM listings WHERE price > 150)          AS above_150,
       (SELECT COUNT(*) FROM listings WHERE NOT (price > 150))    AS not_above_150,
       (SELECT COUNT(*) FROM listings WHERE price IS NULL)        AS no_price,
       (SELECT COUNT(*) FROM listings)                            AS all_listings
"""))
q("""
SELECT id, room_type, price, COALESCE(price, -1) AS price_or_minus_one
FROM listings
WHERE district = 'Mitte'
ORDER BY price          -- ascending: SQLite puts the NULLs first
LIMIT 3
""")

   above_150  not_above_150  no_price  all_listings
0       3576           4865      4335         12776


,id,room_type,price,price_or_minus_one
0,191023,Private room,None,-1
1,251406,Entire home/apt,None,-1
2,415218,Entire home/apt,None,-1


The first three numbers add up to all listings: each listing is above €150, not above €150, or has no price, and only `IS NULL` finds the third group. The second query sorts by price in ascending order, and the listings without a price come first; `ORDER BY price NULLS LAST` would put them at the end. (A query inside parentheses, a **subquery**, follows in block 2; here it just puts four counts into one row.)

### In practice

- **Surveys** distinguish "answered 0" from "did not answer"; storing both as 0 makes the average wrong.
- **Health records** leave a laboratory value empty when the test was not done; treating empty as "normal" or as zero leads to wrong conclusions.

> **Warning.** `WHERE x NOT IN (...)` behaves unexpectedly when the list contains a NULL. Block 2 shows safer ways to find rows without a partner.

## Check your understanding

1. Which column is the primary key of `listings`, and which columns identify a row of `reviews`? What would a database with foreign keys switched on do with a review row for a listing that is not in `listings`?
2. Why does SQLite store dates as text in the form `'YYYY-MM-DD'`, and why does this form sort correctly?
3. Write the clauses `SELECT`, `FROM`, `WHERE`, `ORDER BY`, `LIMIT` in the order the database evaluates them.
4. Why does `SELECT COUNT(*) FROM listings WHERE price = NULL` return 0, and how do you count the listings without a price?
5. In the small example tables, how many rows does an inner join of `listings` and `reviews` return, and how many a left join? Why?

## Further reading

- Codd, E. F. (1970). A relational model of data for large shared data banks. *Communications of the ACM*, 13(6), 377–387. https://doi.org/10.1145/362384.362685
- SQLite. *Datatypes in SQLite*. https://www.sqlite.org/datatype3.html
- SQLite. *SQL as understood by SQLite: SELECT*. https://www.sqlite.org/lang_select.html
- SQLite. *Most widely deployed and used database engine*. https://www.sqlite.org/mostdeployed.html
- Library of Congress. *Recommended formats statement: datasets*. https://www.loc.gov/preservation/resources/rfs/data.html
- Software Carpentry. *Databases and SQL* (lessons with SQLite). https://swcarpentry.github.io/sql-novice-survey/
- [SQLBolt](https://sqlbolt.com/), interactive lessons 1–6 (queries, filtering, sorting).